# EasyOCR bench — SoccerNet-GS

Evaluates **jersey digit OCR** on ground-truth **player** crops from SoccerNet Game State validation clips.

- **Data:** `$SPORTIFY_DATA_ROOT/SoccerNetGS` (see [investigation.md](investigation.md))
- **Clips:** `valid-quick` manifest by default (`SNGS-021` … `023`)
- **Not GS-HOTA:** oracle boxes; measures crop OCR accuracy + speed only

Run all cells after `./setup.sh` and dataset download (`benchmarks/soccernet-gsr/setup-bench.sh`).


In [ ]:
import json
import os
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from statistics import mean

import easyocr
import numpy as np
import yaml
from PIL import Image

try:
    RESAMPLE = Image.Resampling.LANCZOS
except AttributeError:
    RESAMPLE = Image.LANCZOS

# --- paths ---
EXPERIMENT_DIR = Path.cwd()
REPO_ROOT = EXPERIMENT_DIR.parents[3]
MANIFEST_PATH = REPO_ROOT / "benchmarks" / "soccernet-gsr" / "manifests" / "valid-quick.yaml"
DATA_ROOT = Path(os.environ.get("SPORTIFY_DATA_ROOT", Path.home() / "data" / "sportify"))
SOCNET_ROOT = DATA_ROOT / "SoccerNetGS"
SPLIT = "valid"

# --- sampling (tune for smoke vs full) ---
CLIP_IDS = None  # None = load from manifest; or e.g. ["SNGS-021"]
FRAME_STRIDE = 25  # 1 frame/s at 25 FPS
MAX_CROPS_POSITIVE = None  # cap positives for quick runs
MAX_CROPS_NEGATIVE = 200  # null-jersey player boxes; 0 to skip
NEGATIVE_RATIO = 0.25  # subsample negatives vs positives cap

# --- OCR (aligned with throughput-and-errors experiment) ---
IMAGE_SCALE = 4
CROP_PAD_FRAC = 0.12
OCR_KWARGS = {
    "allowlist": "0123456789",
    "mag_ratio": 1.5,
    "contrast_ths": 0.05,
    "adjust_contrast": 0.7,
}
MIN_CONFIDENCE = 0.2
USE_GPU = None
WARMUP = True
PLAYER_CATEGORY_ID = 1


In [ ]:
def parse_jersey(value) -> str | None:
    if value is None:
        return None
    s = str(value).strip()
    if not s or s.lower() in ("null", "none"):
        return None
    digits = re.sub(r"\D", "", s)
    return digits if digits else None


def load_manifest_clips(path: Path) -> list[str]:
    data = yaml.safe_load(path.read_text())
    return [c["id"] for c in data["clips"]]


def load_clip_labels(clip_id: str) -> tuple[dict, dict, list[dict]]:
    clip_dir = SOCNET_ROOT / SPLIT / clip_id
    label_path = clip_dir / "Labels-GameState.json"
    if not label_path.is_file():
        raise FileNotFoundError(label_path)
    raw = json.loads(label_path.read_text())
    version = raw.get("info", {}).get("version", "?")
    if str(version) < "1.3":
        print(f"Warning: {clip_id} version {version} < 1.3")
    image_by_id = {im["image_id"]: im for im in raw["images"]}
    img_dir = clip_dir / raw.get("info", {}).get("im_dir", "img1")
    return image_by_id, {"img_dir": img_dir, "version": version}, raw["annotations"]


@dataclass
class CropSample:
    clip_id: str
    file_name: str
    frame_idx: int
    track_id: int
    gt_jersey: str | None
    bbox: tuple[int, int, int, int]  # x, y, w, h
    is_positive: bool


def build_samples(
    clip_ids: list[str],
    frame_stride: int,
    max_pos: int | None,
    max_neg: int,
    neg_ratio: float,
) -> list[CropSample]:
    positives: list[CropSample] = []
    negatives: list[CropSample] = []

    for clip_id in clip_ids:
        image_by_id, meta, annotations = load_clip_labels(clip_id)
        img_dir = meta["img_dir"]
        frame_order = sorted(image_by_id.values(), key=lambda im: im["file_name"])
        for frame_i, im in enumerate(frame_order):
            if frame_i % frame_stride != 0:
                continue
            file_name = im["file_name"]
            for ann in annotations:
                if ann.get("supercategory") != "object":
                    continue
                if ann.get("category_id") != PLAYER_CATEGORY_ID:
                    continue
                if ann.get("image_id") != im["image_id"]:
                    continue
                bbox = ann.get("bbox_image")
                if not bbox:
                    continue
                gt = parse_jersey((ann.get("attributes") or {}).get("jersey"))
                box = (
                    int(bbox["x"]),
                    int(bbox["y"]),
                    int(bbox["w"]),
                    int(bbox["h"]),
                )
                sample = CropSample(
                    clip_id=clip_id,
                    file_name=file_name,
                    frame_idx=frame_i,
                    track_id=int(ann.get("track_id", -1)),
                    gt_jersey=gt,
                    bbox=box,
                    is_positive=gt is not None,
                )
                if gt is not None:
                    positives.append(sample)
                else:
                    negatives.append(sample)

    if max_pos is not None and len(positives) > max_pos:
        positives = positives[:max_pos]
    if max_neg > 0 and negatives:
        cap = min(max_neg, max(1, int(len(positives) * neg_ratio)))
        negatives = negatives[:cap]

    return positives + negatives


def crop_player(rgb: np.ndarray, box: tuple[int, int, int, int], pad_frac: float) -> np.ndarray:
    x, y, w, h = box
    H, W = rgb.shape[:2]
    px = int(w * pad_frac)
    py = int(h * pad_frac)
    x0 = max(0, x - px)
    y0 = max(0, y - py)
    x1 = min(W, x + w + px)
    y1 = min(H, y + h + py)
    crop = rgb[y0:y1, x0:x1]
    if crop.size == 0:
        return crop
    if IMAGE_SCALE != 1:
        im = Image.fromarray(crop)
        cw, ch = im.size
        im = im.resize((cw * IMAGE_SCALE, ch * IMAGE_SCALE), RESAMPLE)
        crop = np.array(im)
    return crop


def ocr_digits(reader, crop: np.ndarray) -> tuple[str, float]:
    if crop.size == 0:
        return "(none)", 0.0
    t0 = time.perf_counter()
    raw = reader.readtext(crop, detail=1, paragraph=False, **OCR_KWARGS)
    ms = (time.perf_counter() - t0) * 1000.0
    parts = [
        str(x[1]).strip()
        for x in raw
        if float(x[2]) >= MIN_CONFIDENCE and str(x[1]).strip()
    ]
    text = " | ".join(parts) if parts else ""
    digits = re.sub(r"\D", "", text)
    return (digits if digits else "(none)"), ms


def classify_positive(gt: str, pred: str) -> str:
    if pred == "(none)":
        return "miss"
    if pred == gt:
        return "correct"
    return "wrong"


clip_ids = CLIP_IDS or load_manifest_clips(MANIFEST_PATH)
print(f"Repo root:        {REPO_ROOT}")
print(f"SoccerNet root:   {SOCNET_ROOT}")
print(f"Manifest:         {MANIFEST_PATH.name}")
print(f"Clips ({SPLIT}):   {', '.join(clip_ids)}")
print(f"Frame stride:     {FRAME_STRIDE}")


In [ ]:
samples = build_samples(
    clip_ids,
    FRAME_STRIDE,
    MAX_CROPS_POSITIVE,
    MAX_CROPS_NEGATIVE,
    NEGATIVE_RATIO,
)
pos = [s for s in samples if s.is_positive]
neg = [s for s in samples if not s.is_positive]
print(f"Samples: {len(samples)} total ({len(pos)} positive, {len(neg)} negative)")
if not pos:
    raise SystemExit("No positive jersey crops — check dataset path and SPLIT")

by_clip = Counter(s.clip_id for s in pos)
print("Positive crops per clip:", dict(by_clip))
gt_dist = Counter(s.gt_jersey for s in pos)
print("GT jersey distribution (top 10):", gt_dist.most_common(10))


In [ ]:
gpu = USE_GPU
if gpu is None:
    try:
        import torch
        gpu = torch.cuda.is_available()
    except ImportError:
        gpu = False

print("Loading EasyOCR...")
t0 = time.perf_counter()
reader = easyocr.Reader(["en"], gpu=gpu, verbose=False)
print(f"Reader ready in {time.perf_counter() - t0:.1f}s (gpu={gpu})")

# warmup on first positive crop
first_clip = pos[0]
_, meta, _ = load_clip_labels(first_clip.clip_id)
frame_path = meta["img_dir"] / first_clip.file_name
if not frame_path.is_file():
    frame_path = SOCNET_ROOT / SPLIT / first_clip.clip_id / "img1" / first_clip.file_name
rgb = np.array(Image.open(frame_path).convert("RGB"))
if WARMUP:
    _ = ocr_digits(reader, crop_player(rgb, first_clip.bbox, CROP_PAD_FRAC))


In [ ]:
# Cache loaded frames per (clip, file_name)
frame_cache: dict[tuple[str, str], np.ndarray] = {}


def load_frame(sample: CropSample) -> np.ndarray:
    key = (sample.clip_id, sample.file_name)
    if key in frame_cache:
        return frame_cache[key]
    _, meta, _ = load_clip_labels(sample.clip_id)
    path = meta["img_dir"] / sample.file_name
    if not path.is_file():
        path = SOCNET_ROOT / SPLIT / sample.clip_id / "img1" / sample.file_name
    rgb = np.array(Image.open(path).convert("RGB"))
    frame_cache[key] = rgb
    return rgb


results: list[dict] = []
crop_ms: list[float] = []

for i, sample in enumerate(samples):
    rgb = load_frame(sample)
    crop = crop_player(rgb, sample.bbox, CROP_PAD_FRAC)
    pred, ms = ocr_digits(reader, crop)
    crop_ms.append(ms)

    if sample.is_positive:
        outcome = classify_positive(sample.gt_jersey, pred)
        results.append({
            "clip": sample.clip_id,
            "frame": sample.file_name,
            "track": sample.track_id,
            "gt": sample.gt_jersey,
            "pred": pred,
            "outcome": outcome,
            "ms": round(ms, 1),
        })
    else:
        fp = pred != "(none)"
        results.append({
            "clip": sample.clip_id,
            "frame": sample.file_name,
            "track": sample.track_id,
            "gt": "(none)",
            "pred": pred,
            "outcome": "fp" if fp else "tn",
            "ms": round(ms, 1),
        })

pos_results = [r for r in results if r["gt"] != "(none)"]
neg_results = [r for r in results if r["gt"] == "(none)"]

correct = sum(1 for r in pos_results if r["outcome"] == "correct")
miss = sum(1 for r in pos_results if r["outcome"] == "miss")
wrong = sum(1 for r in pos_results if r["outcome"] == "wrong")
fp_neg = sum(1 for r in neg_results if r["outcome"] == "fp")

n_pos = len(pos_results)
n_neg = len(neg_results)

print()
print("=" * 56)
print("EASYOCR — SOCCERNET-GS JERSEY BENCH")
print("=" * 56)
print(f"clips:              {', '.join(clip_ids)}")
print(f"split:              {SPLIT}")
print(f"frame_stride:       {FRAME_STRIDE}")
print(f"image_scale:        {IMAGE_SCALE}  gpu: {gpu}")
print(f"positive crops:     {n_pos}")
print(f"negative crops:     {n_neg}")
print()
print("ACCURACY (player boxes with GT jersey)")
if n_pos:
    print(f"  exact match:      {correct:5d}  ({100 * correct / n_pos:.1f}%)")
    print(f"  miss (no digit):  {miss:5d}  ({100 * miss / n_pos:.1f}%)")
    print(f"  wrong digit:      {wrong:5d}  ({100 * wrong / n_pos:.1f}%)")
print()
print("NULL-JERSEY PLAYERS (should not read a digit)")
if n_neg:
    print(f"  false positive:   {fp_neg:5d}  ({100 * fp_neg / n_neg:.1f}%)")
    print(f"  true negative:    {n_neg - fp_neg:5d}")
else:
    print("  (skipped — set MAX_CROPS_NEGATIVE > 0)")
print()
print("SPEED")
print(f"  mean per crop:    {mean(crop_ms):.1f} ms")
print(f"  crops/s:          {1000.0 / mean(crop_ms):.2f}")
print(f"  frames cached:    {len(frame_cache)}")
print()

# per-clip breakdown
print("PER CLIP (positive only)")
for cid in clip_ids:
    sub = [r for r in pos_results if r["clip"] == cid]
    if not sub:
        continue
    c = sum(1 for r in sub if r["outcome"] == "correct")
    print(f"  {cid}: {c}/{len(sub)} correct ({100 * c / len(sub):.1f}%)")
print()

# confusion pairs
wrong_pairs = Counter((r["gt"], r["pred"]) for r in pos_results if r["outcome"] == "wrong")
if wrong_pairs:
    print("TOP WRONG (gt -> pred)")
    for (gt, pred), cnt in wrong_pairs.most_common(12):
        print(f"  {gt} -> {pred}  x{cnt}")
print()

print("SAMPLE FAILURES (up to 15)")
failures = [r for r in pos_results if r["outcome"] != "correct"][:15]
for r in failures:
    print(f"  {r['clip']} {r['frame']} track={r['track']}  gt={r['gt']} pred={r['pred']}  [{r['outcome']}]")
print("=" * 56)
